# 3 Rephrase

This notebook builds the rephrased branch for the study.

It does four things:
1. Rephrase AI proposals for each requested condition.
2. Rephrase AI reviews for each requested condition.
3. Rephrase shared human proposals once.
4. Rephrase shared human reviews once.

It does not generate new proposals or reviews, prepare embeddings, or run analyses.


In [ ]:
PROPOSAL_CONDITIONS_TO_REPHRASE = []
REVIEW_CONDITIONS_TO_REPHRASE = ['baseline', 'one_at_a_time', 'persona']

REPHRASE_MODEL = 'gemini-3.1-pro-preview'
REPHRASE_TEMPERATURE = 0
MAX_TOKENS_PROPOSALS = 12000
MAX_TOKENS_REVIEWS = 4000
RETRY_DELAYS = [2, 5, 10]
SAVE_PROGRESS_EVERY_N_ROWS = 5
RESUME_OK = True

# Set to a fixed string only if you want to pin new outputs to a custom run id.
RUN_ID = None


In [ ]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from ai_models_interface import AIModelsInterface
from rephrase_pipeline import (
    build_rephrase_job_registry,
    find_project_root,
    load_human_proposal_sources,
    load_human_review_sources,
    locate_latest_ai_proposal_files,
    locate_latest_ai_review_files,
    now_run_id,
    rephrase_ai_proposals_for_condition,
    rephrase_ai_reviews_for_condition,
    rephrase_shared_human_proposals,
    rephrase_shared_human_reviews,
)

PROJECT_ROOT = find_project_root(PROJECT_ROOT)
ai_interface = AIModelsInterface(config_path=str(PROJECT_ROOT / '.env'), override_env=True)
available_models = ai_interface.get_available_models()
resolved_model = ai_interface.resolve_model_name(REPHRASE_MODEL)
if resolved_model not in available_models:
    raise RuntimeError(f'Rephrase model unavailable with current API keys: {resolved_model}')
stage_run_id = RUN_ID or now_run_id()

# --- Registry 1: proposal sources and shared human artifacts ready now ---
# AI reviews remain in a separate registry below because review generation can
# finish later condition by condition.
ai_proposal_sources = locate_latest_ai_proposal_files(PROJECT_ROOT, PROPOSAL_CONDITIONS_TO_REPHRASE)
human_proposal_sources = load_human_proposal_sources(PROJECT_ROOT)
human_review_sources = load_human_review_sources(PROJECT_ROOT)

proposal_registry = build_rephrase_job_registry(
    ai_proposal_sources=ai_proposal_sources,
    ai_review_sources={},
    human_proposal_sources=human_proposal_sources,
    human_review_sources=human_review_sources,
    rephrase_model=resolved_model,
    rephrase_temperature=REPHRASE_TEMPERATURE,
    run_id=stage_run_id,
)

print(f'Project root: {PROJECT_ROOT}')
print(f'Rephrase model: {resolved_model}')
print(f'AI proposal jobs:    {len(proposal_registry["ai_proposal_rephrase_jobs"])} ready={sorted(ai_proposal_sources)}')
print(f'Human proposal jobs: {len(proposal_registry["human_proposal_rephrase_jobs"])}')
print(f'Human review jobs:   {len(proposal_registry["human_review_rephrase_jobs"])}')


In [ ]:
ai_proposal_rephrase_outputs = {}

for job in proposal_registry['ai_proposal_rephrase_jobs']:
    condition = job['condition']
    print(f'\n=== Rephrase AI proposals: {condition} ===')
    result = rephrase_ai_proposals_for_condition(
        project_root=PROJECT_ROOT,
        ai_interface=ai_interface,
        condition=condition,
        source_path=job['source_path'],
        rephrase_model=resolved_model,
        rephrase_temperature=REPHRASE_TEMPERATURE,
        max_tokens=MAX_TOKENS_PROPOSALS,
        retry_delays=RETRY_DELAYS,
        save_every_n_rows=SAVE_PROGRESS_EVERY_N_ROWS,
        resume_ok=RESUME_OK,
        run_id=stage_run_id,
    )
    ai_proposal_rephrase_outputs[condition] = result
    print(f"Source file: {job['source_path']}")
    print(f"Output file: {result['output_path']}")
    if result.get('summary_path'):
        print(f"Summary facts JSON: {result['summary_path']}")
    print(f"Rows: {len(result['rephrased_df'])}")
    if result['qa_issues']:
        print('QA issues:')
        for issue in result['qa_issues'][:10]:
            print(f'  - {issue}')


In [ ]:
# --- Registry 2: AI reviews (run once review generation is complete) ---
# require_all=False builds jobs for whichever review conditions already have a
# complete file and skips the rest, so you can rephrase reviews incrementally.
# Re-run this cell (and the AI-review rephrase cell below) as more conditions
# finish. Running it now with no reviews ready simply yields 0 jobs.
ai_review_sources = locate_latest_ai_review_files(
    PROJECT_ROOT, REVIEW_CONDITIONS_TO_REPHRASE, require_all=False
)
pending_reviews = [c for c in REVIEW_CONDITIONS_TO_REPHRASE if c not in ai_review_sources]

review_registry = build_rephrase_job_registry(
    ai_proposal_sources={},
    ai_review_sources=ai_review_sources,
    human_proposal_sources={},
    human_review_sources={},
    rephrase_model=resolved_model,
    rephrase_temperature=REPHRASE_TEMPERATURE,
    run_id=stage_run_id,
)

print(f'AI review jobs: {len(review_registry["ai_review_rephrase_jobs"])} ready={sorted(ai_review_sources)}')
if pending_reviews:
    print(f'Pending (not yet complete): {pending_reviews}')
    print('Re-run this cell + the AI-review rephrase cell below after those finish generating.')

In [ ]:
ai_review_rephrase_outputs = {}

for job in review_registry['ai_review_rephrase_jobs']:
    condition = job['condition']
    print(f'\n=== Rephrase AI reviews: {condition} ===')
    result = rephrase_ai_reviews_for_condition(
        project_root=PROJECT_ROOT,
        ai_interface=ai_interface,
        condition=condition,
        source_path=job['source_path'],
        rephrase_model=resolved_model,
        rephrase_temperature=REPHRASE_TEMPERATURE,
        max_tokens=MAX_TOKENS_REVIEWS,
        retry_delays=RETRY_DELAYS,
        save_every_n_rows=SAVE_PROGRESS_EVERY_N_ROWS,
        resume_ok=RESUME_OK,
        run_id=stage_run_id,
    )
    ai_review_rephrase_outputs[condition] = result
    print(f"Source file: {job['source_path']}")
    print(f"Output file: {result['output_path']}")
    print(f"Rows: {len(result['rephrased_df'])}")
    if result['qa_issues']:
        print('QA issues:')
        for issue in result['qa_issues'][:10]:
            print(f'  - {issue}')

In [ ]:
# human_proposal_rephrase_outputs = {}
# human_review_rephrase_outputs = {}

# for job in proposal_registry['human_proposal_rephrase_jobs']:
#     cohort = job['cohort']
#     print(f'\n=== Rephrase shared human proposals: {cohort} ===')
#     result = rephrase_shared_human_proposals(
#         project_root=PROJECT_ROOT,
#         ai_interface=ai_interface,
#         source_path=job['source_path'],
#         cohort=cohort,
#         rephrase_model=resolved_model,
#         rephrase_temperature=REPHRASE_TEMPERATURE,
#         max_tokens=MAX_TOKENS_PROPOSALS,
#         retry_delays=RETRY_DELAYS,
#         save_every_n_rows=SAVE_PROGRESS_EVERY_N_ROWS,
#         resume_ok=RESUME_OK,
#         run_id=stage_run_id,
#     )
#     human_proposal_rephrase_outputs[cohort] = result
#     print(f"CSV file: {result['csv_path']}")
#     print(f"JSON file: {result['json_path']}")
#     if result.get('summary_path'):
#         print(f"Summary facts JSON: {result['summary_path']}")
#     print(f"Rows: {len(result['rephrased_df'])}")
#     if result['qa_issues']:
#         print('QA issues:')
#         for issue in result['qa_issues'][:10]:
#             print(f'  - {issue}')

# for job in proposal_registry['human_review_rephrase_jobs']:
#     cohort = job['cohort']
#     print(f'\n=== Rephrase shared human reviews: {cohort} ===')
#     result = rephrase_shared_human_reviews(
#         project_root=PROJECT_ROOT,
#         ai_interface=ai_interface,
#         source_path=job['source_path'],
#         cohort=cohort,
#         rephrase_model=resolved_model,
#         rephrase_temperature=REPHRASE_TEMPERATURE,
#         max_tokens=MAX_TOKENS_REVIEWS,
#         retry_delays=RETRY_DELAYS,
#         save_every_n_rows=SAVE_PROGRESS_EVERY_N_ROWS,
#         resume_ok=RESUME_OK,
#         run_id=stage_run_id,
#     )
#     human_review_rephrase_outputs[cohort] = result
#     print(f"CSV file: {result['csv_path']}")
#     print(f"Rows: {len(result['rephrased_df'])}")
#     if result['qa_issues']:
#         print('QA issues:')
#         for issue in result['qa_issues'][:10]:
#             print(f'  - {issue}')


In [ ]:
summary_rows = []

for condition, result in ai_proposal_rephrase_outputs.items():
    summary_rows.append({
        'artifact_family': 'ai_proposals',
        'scope': condition,
        'rows': len(result['rephrased_df']),
        'output_file': str(result['output_path']),
        'summary_file': str(result.get('summary_path', '')),
        'reused_existing': result['reused_existing'],
        'qa_issue_count': len(result['qa_issues']),
    })

for condition, result in ai_review_rephrase_outputs.items():
    summary_rows.append({
        'artifact_family': 'ai_reviews',
        'scope': condition,
        'rows': len(result['rephrased_df']),
        'output_file': str(result['output_path']),
        'summary_file': '',
        'reused_existing': result['reused_existing'],
        'qa_issue_count': len(result['qa_issues']),
    })

for cohort, result in human_proposal_rephrase_outputs.items():
    summary_rows.append({
        'artifact_family': 'human_proposals',
        'scope': cohort,
        'rows': len(result['rephrased_df']),
        'output_file': str(result['csv_path']),
        'summary_file': str(result.get('summary_path', '')),
        'reused_existing': result['reused_existing'],
        'qa_issue_count': len(result['qa_issues']),
    })

for cohort, result in human_review_rephrase_outputs.items():
    summary_rows.append({
        'artifact_family': 'human_reviews',
        'scope': cohort,
        'rows': len(result['rephrased_df']),
        'output_file': str(result['csv_path']),
        'summary_file': '',
        'reused_existing': result['reused_existing'],
        'qa_issue_count': len(result['qa_issues']),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df
